<a href="https://colab.research.google.com/github/akshatkeshri/D2C-Churn-Part2-Segmentation/blob/main/rfm_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Cell 1
!pip install gdown -q
import gdown
import os
import pandas as pd
import numpy as np

folder_url = 'https://drive.google.com/drive/folders/1PmLapJI1VSDgvl_AxARNKwM1MCd3WFX0'
output_dir = './data'

if not os.path.exists(output_dir):
    print("Fetching dataset...")
    !gdown --folder {folder_url} -O {output_dir}
    print(" Download complete!")

rfm = pd.read_csv('./data/rfm_modeling_snapshot.csv')

print(" Clean Data loaded. Shape:", rfm.shape)

 Clean Data loaded. Shape: (2400, 29)


In [4]:
print("Columns in df_seg:", df_seg.columns.tolist())

Columns in df_seg: ['customer_id', 'snapshot_date', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent', 'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d_x', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d_x', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago', 'churn_next_60d', 'split', 'support_ticket_count', 'abandoned_carts_30d_y', 'sessions_30d_y', 'R_Score']


In [7]:
# Cell 2

rfm['R_Score'] = pd.qcut(rfm['recency_days'], q=4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['frequency_180d'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['monetary_180d'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

def assign_segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    tickets = row['ticket_count_90d']
    carts = row['abandoned_carts_30d']

    if m >= 3 and tickets >= 2:
        return 'High-Value but Unhappy'

    elif r >= 3 and f >= 3 and m >= 3:
        return 'Champions'

    elif r <= 2 and f >= 3 and m >= 3:
        return 'At-Risk Customers'

    elif carts > 0 and f <= 2:
        return 'Discount-Sensitive'

    elif r <= 2 and f <= 2 and m <= 2:
        return 'Dormant Customers'

    else:
        return 'Loyal Customers'

rfm['segment_name'] = rfm.apply(assign_segment, axis=1)

print("\n--- Final Segment Distribution ---")
print(rfm['segment_name'].value_counts())


output_cols = ['customer_id', 'segment_name', 'recency_days', 'frequency_180d', 'monetary_180d', 'ticket_count_90d', 'abandoned_carts_30d']
rfm[output_cols].to_csv('segments.csv', index=False)
print("\n Successfully generated 'segments.csv'")

edge_cases = rfm[(rfm['monetary_180d'] > rfm['monetary_180d'].median()) &
                    (rfm['recency_days'] > 30) &
                    (rfm['sessions_30d'] >= 5)].head(10)

print("\n--- 10 Customer IDs for Manual Review ---")
display(edge_cases[['customer_id', 'segment_name', 'recency_days', 'monetary_180d', 'sessions_30d']])


--- Final Segment Distribution ---
segment_name
Champions                 593
Discount-Sensitive        513
Loyal Customers           497
Dormant Customers         428
At-Risk Customers         324
High-Value but Unhappy     45
Name: count, dtype: int64

 Successfully generated 'segments.csv'

--- 10 Customer IDs for Manual Review ---


,customer_id,segment_name,recency_days,monetary_180d,sessions_30d
4,CUST00005,Champions,38,1781.90,18
13,CUST00014,Champions,51,1382.39,11
14,CUST00015,Champions,35,1523.14,6
25,CUST00026,At-Risk Customers,72,3561.39,14
26,CUST00027,Discount-Sensitive,70,2128.34,11
33,CUST00034,Champions,50,1460.21,7
37,CUST00038,Champions,51,1955.61,5
47,CUST00048,Champions,61,2450.45,12
51,CUST00052,Champions,55,1820.53,7
56,CUST00057,Loyal Customers,136,1713.34,7
